# FoldTrust Benchmark Walkthrough

This notebook demonstrates FoldTrust's benchmark results by loading saved outputs and showing key findings.

**Runtime:** < 30 seconds (no heavy computation; loads pre-computed results)

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set paths relative to notebook location
repo_root = Path.cwd().parent.parent
outputs_dir = repo_root / "benchmarks" / "outputs"

## Layer 2: Structure Accuracy

Load summary statistics for MFE/MEA/centroid predictions vs. reference structures.

In [ ]:
layer2_summary = pd.read_csv(outputs_dir / "layer2" / "layer2_summary_overall.csv")
layer2_summary

MEA achieves the best F1 (0.563) with balanced sensitivity and PPV.

## Layer 3: Calibration

Load calibration metrics and tier PPV.

In [ ]:
with open(outputs_dir / "layer3" / "layer3_summary.json") as f:
    layer3 = json.load(f)

print(f"ECE: {layer3['ece']:.4f}")
print(f"AUROC: {layer3['auroc']:.4f}")
print(f"AUPRC: {layer3['auprc']:.4f}")
print()

tier_summary = pd.DataFrame(layer3["mfe_tier_summary"])
print("MFE Tier PPV:")
tier_summary[["tier", "pooled_ppv", "total_pairs"]]

FIRM tier achieves 67% PPV, showing reliable prediction of true pairs.

## Layer 4: SHAPE Agreement

Load SHAPE correlation metrics for SARS-CoV-2 FSE.

In [ ]:
layer4_metrics = pd.read_csv(outputs_dir / "layer4_shape" / "per_dataset_metrics.csv")
layer4_metrics[["dataset", "spearman", "spearman_pvalue", "auroc"]]

icSHAPE datasets (Zhang) show strongest correlation (ρ = 0.50-0.55).

## Layer 5: Robustness

Load temperature and parameter sweep results.

In [ ]:
temp_retention = pd.read_csv(outputs_dir / "layer5" / "temperature_stem_retention.csv")
temp_pivot = temp_retention.pivot_table(
    index="tier", columns="temperature", values="pooled_retention"
).reindex(["FIRM", "SOFT", "FLOPPY"])
print("Temperature Sweep (pooled retention):")
temp_pivot

In [ ]:
params_retention = pd.read_csv(outputs_dir / "layer5" / "parameters_stem_retention.csv")
params_pivot = params_retention.pivot_table(
    index="tier", columns="parameter_set", values="pooled_retention"
).reindex(["FIRM", "SOFT", "FLOPPY"])[
    ["Turner2004_baseline", "Andronescu2007", "Langdon2018"]
]
print("\nParameter Sweep (pooled retention):")
params_pivot

FIRM tier robust at 25-42°C (95-100% retention). Alternative parameters reduce retention to 60-80%.

## Live Demo: FSE Prediction

Run FoldTrust on the SARS-CoV-2 frameshift element.

In [ ]:
import foldtrust as ft
from foldtrust._core import FoldData

# SARS-CoV-2 FSE sequence
fse_seq = "UUUAAACGGGUUUGCGGUGUAAGUGCAGCCCGUCUUACACCGUGCGGCACAGGCACUAGUACUGAUGUCGUAUACAGGGCU"

# Create FoldData and run pipeline
fd = FoldData(sequence=fse_seq, name="sars2-fse")
ft.tl.run_pipeline(fd)

print(f"Sequence length: {len(fse_seq)} nt")
print(f"MFE: {fd.uns['mfe_energy']:.2f} kcal/mol")
print(f"MFE structure: {fd.structures['mfe']}")
print()

# Show tier summary
tier_counts = fd.obs["tier"].value_counts().reindex(["FIRM", "SOFT", "FLOPPY", "LONELY"])
print("Tier counts (positions):")
print(tier_counts)

In [ ]:
# Plot unpaired probability
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fd.obs["unpaired_prob"], linewidth=1.5)
ax.set_xlabel("Position")
ax.set_ylabel("Unpaired Probability")
ax.set_title("SARS-CoV-2 FSE: Unpaired Probability")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

FoldTrust's benchmark validates:
- **Structure accuracy:** MEA F1 = 0.563 on 600 structures
- **Calibration:** FIRM PPV = 0.674; AUROC = 0.889
- **SHAPE agreement:** ρ = 0.29–0.55 on FSE
- **Robustness:** FIRM 95% retention at 25-42°C

FIRM tier reliably predicts high-confidence stems. See `BENCHMARK.md` for full results.